## Day 5 Daily Task -  Linear vs Ridge vs Lasso Regression
### Dataset: California Housing (built into sklearn, no download needed)
### Goal: preprocess -> train 3 models -> evaluate -> compare -> explain winner

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = fetch_california_housing(as_frame=True)
df = data.frame
print(df.shape)
df.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [3]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]
 
print("Features shape:", X.shape)
print(X.columns.tolist())

Features shape: (20640, 8)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
 
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (16512, 8)
Test shape: (4128, 8)


In [24]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled = scaler.transform(X_test) 

In [40]:
lin_reg = LinearRegression()
ridge = Ridge(alpha=1.0)     # alpha = how strong the penalty is
lasso = Lasso(alpha=0.001)     # smaller alpha here since Lasso penalty is stronger by nature
 
lin_reg.fit(X_train_scaled, y_train)
ridge.fit(X_train_scaled, y_train)
lasso.fit(X_train_scaled, y_train)
 
print("All 3 models trained.")


All 3 models trained.


In [41]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    train_pred = model.predict(X_tr)
    test_pred = model.predict(X_te)
 
    print(f"\n--- {name} ---")
    print(f"Train -> MAE: {mean_absolute_error(y_tr, train_pred):.3f} | "
          f"RMSE: {np.sqrt(mean_squared_error(y_tr, train_pred)):.3f} | "
          f"R2: {r2_score(y_tr, train_pred):.3f}")
    print(f"Test  -> MAE: {mean_absolute_error(y_te, test_pred):.3f} | "
          f"RMSE: {np.sqrt(mean_squared_error(y_te, test_pred)):.3f} | "
          f"R2: {r2_score(y_te, test_pred):.3f}")

In [42]:
evaluate(lin_reg, X_train_scaled, y_train, X_test_scaled, y_test, "Linear Regression")
evaluate(ridge, X_train_scaled, y_train, X_test_scaled, y_test, "Ridge")
evaluate(lasso, X_train_scaled, y_train, X_test_scaled, y_test, "Lasso")


--- Linear Regression ---
Train -> MAE: 0.529 | RMSE: 0.720 | R2: 0.613
Test  -> MAE: 0.533 | RMSE: 0.746 | R2: 0.576

--- Ridge ---
Train -> MAE: 0.529 | RMSE: 0.720 | R2: 0.613
Test  -> MAE: 0.533 | RMSE: 0.746 | R2: 0.576

--- Lasso ---
Train -> MAE: 0.529 | RMSE: 0.720 | R2: 0.613
Test  -> MAE: 0.533 | RMSE: 0.745 | R2: 0.577


In [43]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Linear": lin_reg.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
})
print(coef_df)

      Feature    Linear     Ridge     Lasso
0      MedInc  0.854383  0.854327  0.849140
1    HouseAge  0.122546  0.122624  0.123346
2    AveRooms -0.294410 -0.294210 -0.281273
3   AveBedrms  0.339259  0.339008  0.326050
4  Population -0.002308 -0.002282 -0.001062
5    AveOccup -0.040829 -0.040833 -0.039890
6    Latitude -0.896929 -0.896168 -0.885822
7   Longitude -0.869842 -0.869071 -0.858093


### Totally depends on our alpha values as we need to decide the aggressiveness of shrink 